# UCGLE-F1 — Agent Demo

A local (or API-backed) LLM poses a novel hypothesis, drives a run end-to-end
through the M1–M7 pipeline via MCP tools, and proposes a follow-up scan —
all within the capability envelope (S1–S15 + A1–A6). Reproducible with a
fixed seed.

This notebook is deliberately self-contained: you can re-run it with any
`ChatProvider` (OpenAI, Anthropic, Ollama, llama.cpp, vLLM). Where a provider
is unavailable the cells fall back to the raw tool surface so the physics
pipeline still runs end-to-end.

_References_: V2 (Kawai–Kim 1702.07689), V3 (Kamada et al. 2007.08029),
V4 (2412.09490), V5 (2403.09373).

## 1. Boot the simulator

We use the FastAPI app in-process via `TestClient` so the notebook works in CI.

In [ ]:
from fastapi.testclient import TestClient
from ucgle_f1.m8_agent.server import build_app

app = build_app()
client = TestClient(app)
print({t['name']: t['family'] for t in client.get('/mcp/tools').json()})

## 2. Pose a novel hypothesis (agent-equivalent)

> _Novel question_: does a **Gauss–Bonnet profile with a late plateau**
> (i.e. ξ(φ) flat during the last ∼5 e-folds of inflation) amplify F_GB
> enough to recover η_B without violating the V5 ghost bound (S10)?

We record the hypothesis in the agent memory store so every run and plan
produced below traces back to `conversation_id='conv_demo'`.

In [ ]:
hypothesis_text = (
    'A GB profile with a late plateau (xi flat over the last 5 e-folds) '
    'may amplify F_GB enough to match V2 eta_B without violating S10 '
    '(ghost bound, arXiv:2403.09373).'
)
r = client.post('/mcp/tools/record_hypothesis', json={
    'conversation_id': 'conv_demo',
    'text': hypothesis_text,
})
hyp = r.json()['output']
hypothesis_id = hyp['hypothesisId']
hyp

## 3. Propose an experiment plan with citations

The tool returns a plan scaffold citing the shipped bibliography
(V1–V8). A2 enforces that every arXiv ID resolves against it.

In [ ]:
plan = client.post('/mcp/tools/propose_experiments', json={
    'hypothesis': hypothesis_text,
    'constraints': {'respect': ['S10', 'S14']},
    'budget': {'max_runs': 12},
}).json()['output']
plan['steps'], plan['citations']

## 4. Build a RunConfig and validate it

A6 requires `agent.conversationId` and `agent.hypothesisId`. The
`validate_config` tool returns structured diagnostics before we
burn pipeline time.

In [ ]:
config = {
    'potential': {'kind': 'natural', 'params': {'f_a': 1.0, 'Lambda': 0.001}},
    'couplings': {'xi': 1.0e-3, 'theta_grav': 1.0e-3, 'f_a': 1.0e17,
                  'M_star': 1.0e18, 'M1': 1.0e12, 'S_E2': 1.0},
    'reheating': {'Gamma_phi': 1.0e-6, 'T_reh_GeV': 1.0e13},
    'precision': 'standard',
    'agent': {'conversationId': 'conv_demo', 'hypothesisId': hypothesis_id},
}
diags = client.post('/mcp/tools/validate_config', json={'config': config}).json()['output']
assert diags['valid'], diags

## 5. Issue an approval token and start the run

Mutation tools (`start_run`, `cancel_run`, `scan_parameters`,
`save_plan`) require an `approval_token` with the matching scope.
In production the frontend mints one from a user click; here the
notebook mints one directly via `/api/approvals`.

In [ ]:
tok = client.post('/api/approvals', json={
    'scopes': ['start_run', 'scan_parameters', 'save_plan'],
    'ttl_seconds': 600,
}).json()
tok

In [ ]:
start = client.post('/api/runs', json={
    'config': config,
    'approval_token': tok['tokenId'],
}).json()
run_id = start['runId']
run_id

## 6. Stream the run and inspect the audit

Stream lifecycle events to completion, then fetch the audit report
(S1–S15) and validation report (V1–V8).

In [ ]:
import json, time
while True:
    r = client.get(f'/api/runs/{run_id}').json()
    if r.get('status') == 'completed': break
    time.sleep(0.2)
result = r
print('eta_B =', result['eta_B']['value'])
print('F_GB  =', result['F_GB'])
print('audit:', result['audit']['summary'])

In [ ]:
audit = client.post('/mcp/tools/get_audit', json={'run_id': run_id}).json()['output']
[(c['id'], c['verdict'], c['references']) for c in audit['checks']]

## 7. Agent proposes a follow-up scan

The physicist asks “what would I scan next?” — the agent calls
`suggest_next_parameter_scan`, receives a ScanSpec anchored on the
completed run, and (with approval) executes it.

In [ ]:
scan_spec = client.post('/mcp/tools/suggest_next_parameter_scan',
                        json={'run_id': run_id}).json()['output']
scan_spec['axes']

## 8. Sandbox-only artifact — save the plan

Plans are serialised under `artifacts/agent-sandbox/{conversation_id}/plans/`
with a stable plan_id.

In [ ]:
saved = client.post('/mcp/tools/save_plan', json={
    'plan': plan,
    'approval_token': tok['tokenId'],
    'conversation_id': 'conv_demo',
}).json()
saved

## 9. Patch-proposal round-trip (no auto-apply)

The agent proposes a patch. `dry_run_patch` returns
`auditChecksBroken=[]` when the patch preserves S1–S15; otherwise
A5 rejects the patch before it reaches human review.

In [ ]:
diff = '--- a/backend/src/ucgle_f1/m3_modes/chiral.py\n'\
       '+++ b/backend/src/ucgle_f1/m3_modes/chiral.py\n'\
       '@@ -1,1 +1,1 @@\n'\
       '-_UNITARITY_TOL = 1e-12\n'\
       '+_UNITARITY_TOL = 1e-10\n'
proposal = client.post('/mcp/tools/propose_patch', json={
    'conversation_id': 'conv_demo',
    'target_path': 'backend/src/ucgle_f1/m3_modes/chiral.py',
    'rationale': 'relax unitarity tolerance (expect rejection by A5)',
    'unified_diff': diff,
}).json()['output']
proposal

## 10. Reproducibility

Rerun the notebook with the same `conv_demo` and `seed=0` — A4
asserts that identical tool-call sequences and numerical outputs
result.